In [6]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sepsisSimDiabetes.State import State
from sepsisSimDiabetes.Action import Action
from sepsisSimDiabetes.MDP import MDP

In [381]:
class FactorizedDynamics(nn.Module):
    def __init__(self, state_dims, action_dim, emb_dim=32, hidden=128):
        """
        state_dims: list of int, the number of categories for each of the 8 state dims [3, 3, 2, 5, 2, 2, 2, 2]
        action_dim: 3 (binary each)
        """
        super().__init__()
        # embeddings for each state dimension
        self.state_embs = nn.ModuleList([
            nn.Embedding(num_cat, emb_dim) for num_cat in state_dims
        ])
        # action embedding (or just treat as continuous)
        self.act_emb = nn.Linear(action_dim, emb_dim)
        # shared MLP core
        self.fc1 = nn.Linear(emb_dim*(len(state_dims)+1), hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        # separate heads to predict next‐state distribution for each dim
        self.heads = nn.ModuleList([
            nn.Linear(hidden, num_cat) for num_cat in state_dims
        ])

    def forward(self, s, a):
        # s: LongTensor [B, 8], a: FloatTensor [B,3]
        embs = [emb(s[:,i]) for i,emb in enumerate(self.state_embs)]
        embs.append(self.act_emb(a))
        h = torch.cat(embs, dim=-1)
        h = F.relu(self.fc1(h))
        h = F.relu(self.fc2(h))
        # outputs: list of logits [B, num_cat_i]
        logits = [head(h) for head in self.heads]
        return logits

In [771]:
env = MDP()
pi_b = [0.1, 0.3, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
pi_e = [0.8, 0.1, 0.01, 0.01, 0.02, 0.02, 0.02, 0.02]
H = 20
gamma = 0.9
n_actions = Action.NUM_ACTIONS_TOTAL
n_components = 2
# set random seed
np.random.seed(0)
torch.manual_seed(0)

In [772]:
# 15 trajectories to train the model
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_traj = []
_rewards = []
for _ in range(10):
    reward = 0
    trajectory = []
    action_vec = []
    action_idx = []
    reward_traj = []
    s = env.get_new_state(idx_type="full")
    env.state = s
    for t in range(H):
        record_s = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        trajectory.append(record_s)
        a_idx = np.random.choice(n_actions, p=pi_b)
        a = Action(action_idx=a_idx)
        r = env.transition(a)
        record_a = a.get_action_vec().reshape(-1)
        action_vec.append(record_a)
        action_idx.append(a_idx)
        reward_traj.append(r)
        reward += r * gamma**t
        if r != 0: # end of episode
            break
        s = env.state
    _trajectories.append(trajectory)
    _actions_vec.append(action_vec)
    _actions_idx.append(action_idx)
    _rewards_traj.append(reward_traj)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_traj),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('data/train_0_o.pkl', 'wb') as f:
    pickle.dump(data, f)

In [773]:
train_0_o = pickle.load(open("data/train_0_o.pkl", "rb"))
train_0_o_trajs = train_0_o['trajectories']
train_0_o_actions_vec = train_0_o['actions_vec']
train_0_o_actions_idx = train_0_o['actions_idx']
train_0_o_rewards_trajs = train_0_o['rewards_trajectories']
train_0_o_rewards = train_0_o['rewards']

# convert to batch data
def convert_to_batch_data(trajectories, actions, rewards):
    batch_data = []
    for traj, action, reward in zip(trajectories, actions, rewards):
        for t in range(len(traj)-1):
            s = traj[t]
            a = action[t]
            r = reward[t]
            s_next = traj[t+1]
            batch_data.append((s, a, r, s_next))
    return batch_data
batch_data = convert_to_batch_data(train_0_o_trajs, train_0_o_actions_vec, train_0_o_rewards_trajs)

state_dims = [3, 3, 2, 5, 2, 2, 2, 2] # fully observable, no confoundings
action_dim = 3
T = FactorizedDynamics(state_dims, action_dim)
optimizer = optim.Adam(T.parameters(), lr=0.001)
num_epochs = 100
batch_size = 32
loss_fn = nn.CrossEntropyLoss()

In [774]:
# 15 trajectories to calibrate
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_traj = []
_rewards = []
for _ in range(10):
    reward = 0
    trajectory = []
    action_vec = []
    action_idx = []
    reward_traj = []
    s = env.get_new_state(idx_type="full")
    env.state = s
    for t in range(H):
        record_s = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        trajectory.append(record_s)
        a_idx = np.random.choice(n_actions, p=pi_b)
        a = Action(action_idx=a_idx)
        r = env.transition(a)
        record_a = a.get_action_vec().reshape(-1)
        action_vec.append(record_a)
        action_idx.append(a_idx)
        reward_traj.append(r)
        reward += r * gamma**t
        if r != 0: # end of episode
            break
        s = env.state
    _trajectories.append(trajectory)
    _actions_vec.append(action_vec)
    _actions_idx.append(action_idx)
    _rewards_traj.append(reward_traj)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_traj),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('data/cal_0_o.pkl', 'wb') as f:
    pickle.dump(data, f)

In [775]:
cal_0_o = pickle.load(open("data/cal_0_o.pkl", "rb"))
cal_0_o_trajs = cal_0_o['trajectories']
cal_0_o_actions_vec = cal_0_o['actions_vec']
cal_0_o_actions_idx = cal_0_o['actions_idx']
cal_0_o_rewards_trajs = cal_0_o['rewards_trajectories']
cal_0_o_rewards = cal_0_o['rewards']

# merge the two datasets
whole_0_o_trajs = np.concatenate((train_0_o_trajs, cal_0_o_trajs), axis=0)
whole_0_o_actions_vec = np.concatenate((train_0_o_actions_vec, cal_0_o_actions_vec), axis=0)
whole_0_o_actions_idx = np.concatenate((train_0_o_actions_idx, cal_0_o_actions_idx), axis=0)
whole_0_o_rewards_trajs = np.concatenate((train_0_o_rewards_trajs, cal_0_o_rewards_trajs), axis=0)
whole_0_o_rewards = np.concatenate((train_0_o_rewards, cal_0_o_rewards), axis=0)

In [776]:
whole_weights = np.zeros(len(whole_0_o_trajs))
for i in range(len(whole_0_o_trajs)):
    w = 1
    for t in range(len(whole_0_o_trajs[i])):
        w *= pi_e[int(whole_0_o_actions_idx[i][t])] / pi_b[int(whole_0_o_actions_idx[i][t])]
    whole_weights[i] = w

In [790]:
alpha = 0.05
z = norm.ppf(1 - alpha/2)

In [791]:
# IS CLT
V_IS = np.zeros(len(whole_0_o_trajs))
for i in range(len(whole_0_o_trajs)):
    V_IS[i] = whole_weights[i] * whole_0_o_rewards[i]
V_IS_mean = np.mean(V_IS)
V_IS_var = np.var(V_IS)
V_IS_lb = V_IS_mean - z * np.sqrt(V_IS_var / len(V_IS))
V_IS_ub = V_IS_mean + z * np.sqrt(V_IS_var / len(V_IS))
print('V_IS:', V_IS_mean)
print('V_IS_LB:', V_IS_lb)
print('V_IS_UB:', V_IS_ub)

V_IS: 0.33736605291864785
V_IS_LB: -0.4347276665100339
V_IS_UB: 1.1094597723473296


In [792]:
# DR CLT

# train a DQN using the batch data
class DQN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim) 
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, s, a):
        x = torch.cat((s, a), dim=1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
Q = DQN(input_dim=8+3, hidden_dim=128, output_dim=1)

optimizer = optim.Adam(Q.parameters(), lr=0.001)
criterion = nn.MSELoss()
num_epochs = 100
batch_size = 16
for epoch in range(num_epochs):
    np.random.shuffle(batch_data)
    for i in range(0, len(batch_data), batch_size):
        batch = batch_data[i:i + batch_size]
        s = torch.tensor([b[0] for b in batch], dtype=torch.float32)
        a = torch.tensor([b[1] for b in batch], dtype=torch.float32)
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
        s1 = torch.tensor([b[3] for b in batch], dtype=torch.float32)
        # print(s.shape, a.shape, r.shape, s1.shape)
        q_value = Q(s, a)
        # print(q_value.shape)
        V_pi_e = 0
        for a in range(n_actions):
            prob = torch.tensor([pi_e[a]], dtype=torch.float32).expand(s1.shape[0], 1)
            # get the action embedding
            a = torch.tensor([Action(action_idx=a).get_action_vec()], dtype=torch.float32).reshape(1, -1).expand(s1.shape[0], 3)
            V_pi_e += prob * Q(s1, a)
        
        target_q_value = r.unsqueeze(1) + gamma * V_pi_e
        loss = criterion(q_value, target_q_value)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [0/100], Loss: 0.0025
Epoch [10/100], Loss: 0.0001
Epoch [20/100], Loss: 0.0001
Epoch [30/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [70/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [90/100], Loss: 0.0000


In [793]:
V_DR_step = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    v = 0
    for t in range(len(cal_0_o_trajs[i])):
        s = cal_0_o_trajs[i][t]
        a_vec = cal_0_o_actions_vec[i][t]
        a_idx = cal_0_o_actions_idx[i][t]
        r = cal_0_o_rewards_trajs[i][t]
        V_s = 0
        s = torch.tensor(s, dtype=torch.float32).reshape(1, -1)
        a_vec = torch.tensor(a_vec, dtype=torch.float32).reshape(1, -1)
        for a1 in range(n_actions):
            # get the action embedding
            tensor_a1 = torch.tensor([Action(action_idx=a1).get_action_vec()], dtype=torch.float32).reshape(1, -1)
            V_s_a1 = Q(s, tensor_a1).detach().numpy()
            V_s += pi_e[a1] * float(V_s_a1)
        Q_s_a = Q(s, a_vec).detach().numpy()
        Q_s_a = float(Q_s_a)
        w = pi_e[a_idx] / pi_b[a_idx]
        v = V_s + w * (r + gamma * v - Q_s_a)
    V_DR_step[i] = v

# calculate the average of V_DR
V_DR_step_mean = np.mean(V_DR_step)
V_DR_step_var = np.var(V_DR_step)
V_DR_step_lb = V_DR_step_mean - z * np.sqrt(V_DR_step_var / len(V_DR_step))
V_DR_step_ub = V_DR_step_mean + z * np.sqrt(V_DR_step_var / len(V_DR_step))
print('V_DR:', V_DR_step_mean)
print('V_DR_LB:', V_DR_step_lb)
print('V_DR_UB:', V_DR_step_ub)

V_DR: 0.5699399684168751
V_DR_LB: -1.012800551448165
V_DR_UB: 2.1526804882819155


/var/folders/yc/lqgm3q450pb2g1j5qwbhhqsm0000gn/T/ipykernel_4723/3066623568.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  V_s += pi_e[a1] * float(V_s_a1)
/var/folders/yc/lqgm3q450pb2g1j5qwbhhqsm0000gn/T/ipykernel_4723/3066623568.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Q_s_a = float(Q_s_a)


In [794]:
# IS Bootstrap
bootstrap_samples = len(whole_0_o_trajs) * 10
V_IS_bootstrap = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    idx = np.random.randint(0, len(whole_0_o_trajs), size=len(whole_0_o_trajs))
    V_IS_bootstrap[i] = np.mean(V_IS[idx])
V_IS_bootstrap_mean = np.mean(V_IS_bootstrap)
# get the alpha quantile and (1-alpha) quantile
V_IS_bootstrap_alpha = np.quantile(V_IS_bootstrap, alpha/2)
V_IS_bootstrap_1_alpha = np.quantile(V_IS_bootstrap, 1 - alpha/2)
print('V_IS_bootstrap:', V_IS_bootstrap_mean)
print('V_IS_bootstrap_lb:', V_IS_bootstrap_alpha)
print('V_IS_bootstrap_1_ub:', V_IS_bootstrap_1_alpha)

V_IS_bootstrap: 0.3285304882708966
V_IS_bootstrap_lb: -0.11123464276493383
V_IS_bootstrap_1_ub: 1.1721558482048335


In [795]:
# train a transition model
def train_T(data, epochs, batch_size):
    state_dims = [3, 3, 2, 5, 2, 2, 2, 2]
    action_dim = 3

    batch_data = []
    train_0_o_trajs = data['trajectories']
    train_0_o_actions_vec = data['actions_vec']
    train_0_o_actions_idx = data['actions_idx']
    train_0_o_rewards_trajs = data['rewards_trajectories']
    train_0_o_rewards = data['rewards']
    for i in range(len(train_0_o_trajs)):
        for t in range(len(train_0_o_trajs[i]) - 1):
            s = train_0_o_trajs[i][t]
            a_vec = train_0_o_actions_vec[i][t]
            a_idx = train_0_o_actions_idx[i][t]
            r = train_0_o_rewards_trajs[i][t]
            s1 = train_0_o_trajs[i][t + 1]
            batch_data.append((s, a_vec, r, s1))
            
    T = FactorizedDynamics(state_dims, action_dim)
    optimizer_T = optim.Adam(T.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    
    num_epochs = epochs
    batch_size = batch_size

    for epoch in range(num_epochs):
        # shuffle the data
        np.random.shuffle(batch_data)
        for i in range(0, len(batch_data), batch_size):
            batch = batch_data[i:i + batch_size]
            s = torch.LongTensor([b[0] for b in batch])
            a = torch.FloatTensor([b[1] for b in batch])
            r = torch.FloatTensor([b[2] for b in batch])
            s1 = torch.LongTensor([b[3] for b in batch])

            optimizer_T.zero_grad()
            logits = T(s, a)
            loss = 0
            for j in range(len(state_dims)):
                loss += loss_fn(logits[j], s1[:, j])
            loss.backward()
            optimizer_T.step()

        if epoch % int(num_epochs / 10) == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Loss T: {loss.item():.4f}')
    return T

In [796]:
# MB Bootstrap
bootstrap_samples = 10
V_MB_bootstrap = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    # sample 20 trajectories from whole_0_o_trajs
    idx = np.random.randint(0, len(whole_0_o_trajs), len(whole_0_o_trajs))
    data = {
        'trajectories': whole_0_o_trajs[idx],
        'actions_vec': whole_0_o_actions_vec[idx],
        'actions_idx': whole_0_o_actions_idx[idx],
        'rewards_trajectories': whole_0_o_rewards_trajs[idx],
        'rewards': whole_0_o_rewards[idx]
    }
    T = train_T(data, epochs=100, batch_size=32)
    # rollout 1000 trajectories
    V = 0
    for j in range(1000):
        s = env.get_new_state(idx_type="full")
        env.state = s
        reward = 0
        for h in range(H):
            s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
            a_idx = np.random.choice(n_actions, p=pi_e)
            a = Action(action_idx=a_idx)
            r = env.transition(a)
            reward += r * gamma**h
            if r != 0:
                break
            tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
            tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
            with torch.no_grad():
                logits = T(tensor_s, tensor_a)
                probs = [F.softmax(log, dim=-1) for log in logits]
                # sample each dimension independently
                s1 = torch.stack([
                    torch.multinomial(p, num_samples=1).squeeze(-1)
                    for p in probs
                ], dim=1).numpy()[0]
            state_categs = s1[:-1]
            diabetic_idx = s1[-1]
            env.state = State(
                state_categs=state_categs,
                diabetic_idx=diabetic_idx,
            )
            s = env.state
        V += reward
    V_MB_bootstrap[i] = V / 1000

Epoch [0/100], Loss T: 6.3663
Epoch [10/100], Loss T: 1.8381
Epoch [20/100], Loss T: 1.0545
Epoch [30/100], Loss T: 0.3720
Epoch [40/100], Loss T: 0.2397
Epoch [50/100], Loss T: 0.1008
Epoch [60/100], Loss T: 0.0937
Epoch [70/100], Loss T: 0.0561
Epoch [80/100], Loss T: 0.0330
Epoch [90/100], Loss T: 0.0260
Epoch [0/100], Loss T: 5.9807
Epoch [10/100], Loss T: 2.4945
Epoch [20/100], Loss T: 1.5292
Epoch [30/100], Loss T: 0.8525
Epoch [40/100], Loss T: 0.4912
Epoch [50/100], Loss T: 0.5244
Epoch [60/100], Loss T: 0.3104
Epoch [70/100], Loss T: 0.2723
Epoch [80/100], Loss T: 0.2200
Epoch [90/100], Loss T: 0.0851
Epoch [0/100], Loss T: 5.8653
Epoch [10/100], Loss T: 2.6511
Epoch [20/100], Loss T: 0.9941
Epoch [30/100], Loss T: 1.0456
Epoch [40/100], Loss T: 0.6282
Epoch [50/100], Loss T: 0.8124
Epoch [60/100], Loss T: 0.0569
Epoch [70/100], Loss T: 0.0750
Epoch [80/100], Loss T: 0.3129
Epoch [90/100], Loss T: 0.4014
Epoch [0/100], Loss T: 6.1383
Epoch [10/100], Loss T: 1.9195
Epoch [20/10

In [797]:
V_MB_bootstrap_mean = np.mean(V_MB_bootstrap)
# get the alpha quantile and (1-alpha) quantile
V_MB_bootstrap_alpha = np.quantile(V_MB_bootstrap, alpha/2)
V_MB_bootstrap_1_alpha = np.quantile(V_MB_bootstrap, 1 - alpha/2)
print('V_MB_bootstrap:', V_MB_bootstrap_mean)
print('V_MB_bootstrap_lb:', V_MB_bootstrap_alpha)
print('V_MB_bootstrap_ub:', V_MB_bootstrap_1_alpha)

V_MB_bootstrap: -0.39923522414178103
V_MB_bootstrap_lb: -0.5028656957136028
V_MB_bootstrap_ub: -0.28137381730878036


In [798]:
# train a transition model using whole_0_o
whole_0_o = {
    'trajectories': whole_0_o_trajs,
    'actions_vec': whole_0_o_actions_vec,
    'actions_idx': whole_0_o_actions_idx,
    'rewards_trajectories': whole_0_o_rewards_trajs,
    'rewards': whole_0_o_rewards
}
T = train_T(whole_0_o, epochs=100, batch_size=16)

Epoch [0/100], Loss T: 5.0582
Epoch [10/100], Loss T: 1.7526
Epoch [20/100], Loss T: 1.3353
Epoch [30/100], Loss T: 0.6798
Epoch [40/100], Loss T: 0.2470
Epoch [50/100], Loss T: 0.6228
Epoch [60/100], Loss T: 0.3413
Epoch [70/100], Loss T: 0.5029
Epoch [80/100], Loss T: 0.2710
Epoch [90/100], Loss T: 0.1926


In [799]:
# generate augmented data
augmeng_samples = 5000
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []

for i in range(augmeng_samples):
    reward = 0
    trajectory = []
    action_vec = []
    action_idx = []
    reward_traj = []
    s = env.get_new_state(idx_type="full")
    env.state = s
    for h in range(H):
        s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        trajectory.append(s_vec)
        a_idx = np.random.choice(n_actions, p=pi_e)
        a = Action(action_idx=a_idx)
        record_a = a.get_action_vec().reshape(-1)
        action_vec.append(record_a)
        action_idx.append(a_idx)
        r = env.transition(a)
        reward_traj.append(r)
        reward += r * gamma**h
        if r != 0:
            break
        tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
        tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
        with torch.no_grad():
            logits = T(tensor_s, tensor_a)
            probs = [F.softmax(log, dim=-1) for log in logits]
            # sample each dimension independently
            s1 = torch.stack([
                torch.multinomial(p, num_samples=1).squeeze(-1)
                for p in probs
            ], dim=1).numpy()[0]
        state_categs = s1[:-1]
        diabetic_idx = s1[-1]
        env.state = State(
            state_categs=state_categs,
            diabetic_idx=diabetic_idx,
        )
        s = env.state
    _trajectories.append(trajectory)
    _actions_vec.append(action_vec)
    _actions_idx.append(action_idx)
    _rewards_trajectories.append(reward_traj)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('data/aug_0_diff.pkl', 'wb') as f:
    pickle.dump(data, f)

In [800]:
aug_0_diff = pickle.load(open("data/aug_0_diff.pkl", "rb"))
aug_0_diff_trajs = aug_0_diff['trajectories']
aug_0_diff_actions_vec = aug_0_diff['actions_vec']
aug_0_diff_actions_idx = aug_0_diff['actions_idx']
aug_0_diff_rewards_trajs = aug_0_diff['rewards_trajectories']
aug_0_diff_rewards = aug_0_diff['rewards']

# merge aug and whole
aug_0_o_trajs = np.concatenate((aug_0_diff_trajs, whole_0_o_trajs), axis=0)
aug_0_o_actions_vec = np.concatenate((aug_0_diff_actions_vec, whole_0_o_actions_vec), axis=0)
aug_0_o_actions_idx = np.concatenate((aug_0_diff_actions_idx, whole_0_o_actions_idx), axis=0)
aug_0_o_rewards_trajs = np.concatenate((aug_0_diff_rewards_trajs, whole_0_o_rewards_trajs), axis=0)
aug_0_o_rewards = np.concatenate((aug_0_diff_rewards, whole_0_o_rewards), axis=0)



In [801]:
# augment IS CLT
aug_weights = np.zeros(len(aug_0_diff_trajs))
for i in range(len(aug_0_diff_trajs)):
    w = 1
    for t in range(len(aug_0_diff_trajs[i])):
        w *= pi_e[int(aug_0_diff_actions_idx[i][t])] / pi_b[int(aug_0_diff_actions_idx[i][t])]
    aug_weights[i] = w

V_IS_aug = np.zeros(len(aug_0_diff_trajs))
for i in range(len(aug_0_diff_trajs)):
    V_IS_aug[i] = aug_weights[i] * aug_0_diff_rewards[i]
V_IS_aug_mean = np.mean(V_IS_aug)
V_IS_aug_var = np.var(V_IS_aug)
V_IS_aug_lb = V_IS_aug_mean - z * np.sqrt(V_IS_aug_var / len(V_IS_aug))
V_IS_aug_ub = V_IS_aug_mean + z * np.sqrt(V_IS_aug_var / len(V_IS_aug))
print('V_IS_aug:', V_IS_aug_mean)
print('V_IS_aug_LB:', V_IS_aug_lb)
print('V_IS_aug_UB:', V_IS_aug_ub)

V_IS_aug: -3217548679351.848
V_IS_aug_LB: -12071960299995.822
V_IS_aug_UB: 5636862941292.127


In [802]:
# augment DR CLT

# DR CLT

# seperate the data aug_0_diff into train and test (1:4)
train_0_diff_trajs = aug_0_diff_trajs[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_actions_vec = aug_0_diff_actions_vec[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_actions_idx = aug_0_diff_actions_idx[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_rewards_trajs = aug_0_diff_rewards_trajs[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_rewards = aug_0_diff_rewards[:int(len(aug_0_diff_trajs) * 0.2)]
cal_0_diff_trajs = aug_0_diff_trajs[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_actions_vec = aug_0_diff_actions_vec[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_actions_idx = aug_0_diff_actions_idx[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_rewards_trajs = aug_0_diff_rewards_trajs[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_rewards = aug_0_diff_rewards[int(len(aug_0_diff_trajs) * 0.2):]


# make batch data (s, a, r, s1)
batch_data = []
for i in range(len(train_0_diff_trajs)):
    for t in range(len(train_0_diff_trajs[i]) - 1):
        s = train_0_diff_trajs[i][t]
        a = train_0_diff_actions_vec[i][t]
        r = train_0_diff_rewards_trajs[i][t]
        s1 = train_0_diff_trajs[i][t + 1]
        batch_data.append((s, a, r, s1))

aug_Q = DQN(input_dim=8+3, hidden_dim=128, output_dim=1)

optimizer = optim.Adam(aug_Q.parameters(), lr=0.001)
criterion = nn.MSELoss()
num_epochs = 100
batch_size = 64
for epoch in range(num_epochs):
    np.random.shuffle(batch_data)
    for i in range(0, len(batch_data), batch_size):
        batch = batch_data[i:i + batch_size]
        s = torch.tensor([b[0] for b in batch], dtype=torch.float32)
        a = torch.tensor([b[1] for b in batch], dtype=torch.float32)
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
        s1 = torch.tensor([b[3] for b in batch], dtype=torch.float32)
        # print(s.shape, a.shape, r.shape, s1.shape)
        q_value = aug_Q(s, a)
        # print(q_value.shape)
        V_pi_e = 0
        for a in range(n_actions):
            prob = torch.tensor([pi_e[a]], dtype=torch.float32).expand(s1.shape[0], 1)
            # get the action embedding
            a = torch.tensor([Action(action_idx=a).get_action_vec()], dtype=torch.float32).reshape(1, -1).expand(s1.shape[0], 3)
            V_pi_e += prob * aug_Q(s1, a)
        
        target_q_value = r.unsqueeze(1) + gamma * V_pi_e
        loss = criterion(q_value, target_q_value)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [0/100], Loss: 0.0000
Epoch [10/100], Loss: 0.0000
Epoch [20/100], Loss: 0.0000
Epoch [30/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [70/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [90/100], Loss: 0.0000


In [803]:
V_DR_step_aug = np.zeros(len(cal_0_diff_trajs))
for i in range(len(cal_0_diff_trajs)):
    v = 0
    for t in range(len(cal_0_diff_trajs[i])):
        s = cal_0_diff_trajs[i][t]
        a_vec = cal_0_diff_actions_vec[i][t]
        a_idx = cal_0_diff_actions_idx[i][t]
        r = cal_0_diff_rewards_trajs[i][t]
        V_s = 0
        s = torch.tensor(s, dtype=torch.float32).reshape(1, -1)
        a_vec = torch.tensor(a_vec, dtype=torch.float32).reshape(1, -1)
        for a1 in range(n_actions):
            # get the action embedding
            tensor_a1 = torch.tensor([Action(action_idx=a1).get_action_vec()], dtype=torch.float32).reshape(1, -1)
            V_s_a1 = aug_Q(s, tensor_a1).detach().numpy()
            V_s += pi_e[a1] * float(V_s_a1)
        Q_s_a = aug_Q(s, a_vec).detach().numpy()
        Q_s_a = float(Q_s_a)
        w = pi_e[a_idx] / pi_b[a_idx]
        v = V_s + w * (r + gamma * v - Q_s_a)
    V_DR_step_aug[i] = v

# calculate the average of V_DR
V_DR_step_aug_mean = np.mean(V_DR_step_aug)
V_DR_step_aug_var = np.var(V_DR_step_aug)
V_DR_step_aug_lb = V_DR_step_aug_mean - z * np.sqrt(V_DR_step_aug_var / len(V_DR_step_aug))
V_DR_step_aug_ub = V_DR_step_aug_mean + z * np.sqrt(V_DR_step_aug_var / len(V_DR_step_aug))
print('V_DR_aug:', V_DR_step_aug_mean)
print('V_DR_aug_LB:', V_DR_step_aug_lb)
print('V_DR_aug_UB:', V_DR_step_aug_ub)

/var/folders/yc/lqgm3q450pb2g1j5qwbhhqsm0000gn/T/ipykernel_4723/3184275655.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  V_s += pi_e[a1] * float(V_s_a1)
/var/folders/yc/lqgm3q450pb2g1j5qwbhhqsm0000gn/T/ipykernel_4723/3184275655.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Q_s_a = float(Q_s_a)


V_DR_aug: -10334657661.283428
V_DR_aug_LB: -45335092136.43077
V_DR_aug_UB: 24665776813.86392


In [804]:
# augment IS Bootstrap
bootstrap_samples = 200
V_IS_bootstrap_aug = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    idx = np.random.randint(0, len(aug_0_diff_trajs), size=len(aug_0_diff_trajs))
    V_IS_bootstrap_aug[i] = np.mean(V_IS_aug[idx])
V_IS_bootstrap_aug_mean = np.mean(V_IS_bootstrap_aug)
# get the alpha quantile and (1-alpha) quantile
V_IS_bootstrap_aug_alpha = np.quantile(V_IS_bootstrap_aug, alpha/2)
V_IS_bootstrap_aug_1_alpha = np.quantile(V_IS_bootstrap_aug, 1 - alpha/2)
print('V_IS_bootstrap_aug:', V_IS_bootstrap_aug_mean)
print('V_IS_bootstrap_aug_lb:', V_IS_bootstrap_aug_alpha)
print('V_IS_bootstrap_aug_ub:', V_IS_bootstrap_aug_1_alpha)

V_IS_bootstrap_aug: -2779794815549.2036
V_IS_bootstrap_aug_lb: -12008373099585.102
V_IS_bootstrap_aug_ub: 2558646773363.4297


In [805]:
# # augment MB Bootstrap
# # augment MB Bootstrap
# bootstrap_samples = 10
# V_MB_bootstrap_aug = np.zeros(bootstrap_samples)
# for i in range(bootstrap_samples):
#     idx = np.random.randint(0, len(aug_0_diff_trajs), len(aug_0_diff_trajs))
#     data = {
#         'trajectories': aug_0_diff_trajs[idx],
#         'actions_vec': aug_0_diff_actions_vec[idx],
#         'actions_idx': aug_0_diff_actions_idx[idx],
#         'rewards_trajectories': aug_0_diff_rewards_trajs[idx],
#         'rewards': aug_0_diff_rewards[idx]
#     }
#     T = train_T(data, epochs=20, batch_size=256)
#     # rollout 1000 trajectories
#     V = 0
#     for j in range(1000):
#         s = env.get_new_state(idx_type="full")
#         env.state = s
#         reward = 0
#         for h in range(H):
#             s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
#             a_idx = np.random.choice(n_actions, p=pi_e)
#             a = Action(action_idx=a_idx)
#             r = env.transition(a)
#             reward += r * gamma**h
#             if r != 0:
#                 break
#             tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
#             tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
#             with torch.no_grad():
#                 logits = T(tensor_s, tensor_a)
#                 probs = [F.softmax(log, dim=-1) for log in logits]
#                 # sample each dimension independently
#                 s1 = torch.stack([
#                     torch.multinomial(p, num_samples=1).squeeze(-1)
#                     for p in probs
#                 ], dim=1).numpy()[0]
#             state_categs = s1[:-1]
#             diabetic_idx = s1[-1]
#             env.state = State(
#                 state_categs=state_categs,
#                 diabetic_idx=diabetic_idx,
#             )
#             s = env.state
#         V += reward
#     V_MB_bootstrap_aug[i] = V / 1000
# V_MB_bootstrap_aug_mean = np.mean(V_MB_bootstrap_aug)
# # get the alpha quantile and (1-alpha) quantile
# V_MB_bootstrap_aug_alpha = np.quantile(V_MB_bootstrap_aug, alpha/2)
# V_MB_bootstrap_aug_1_alpha = np.quantile(V_MB_bootstrap_aug, 1 - alpha/2)
# print('V_MB_bootstrap_aug:', V_MB_bootstrap_aug_mean)
# print('V_MB_bootstrap_aug_lb:', V_MB_bootstrap_aug_alpha)
# print('V_MB_bootstrap_aug_ub:', V_MB_bootstrap_aug_1_alpha)

In [806]:
# DR-PPI
# train a transition model using train_0_o
T = train_T(train_0_o, epochs=100, batch_size=16)
# generate 5000 trajectories, serve as the first term in DR-PPI N_f=5000
N_f = 1000
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(N_f):
    reward = 0
    trajectory = []
    action_vec = []
    action_idx = []
    reward_traj = []
    s = env.get_new_state(idx_type="full")
    env.state = s
    for h in range(H):
        s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        trajectory.append(s_vec)
        a_idx = np.random.choice(n_actions, p=pi_e)
        a = Action(action_idx=a_idx)
        record_a = a.get_action_vec().reshape(-1)
        action_vec.append(record_a)
        action_idx.append(a_idx)
        r = env.transition(a)
        reward_traj.append(r)
        reward += r * gamma**h
        if r != 0:
            break
        tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
        tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
        with torch.no_grad():
            logits = T(tensor_s, tensor_a)
            probs = [F.softmax(log, dim=-1) for log in logits]
            # sample each dimension independently
            s1 = torch.stack([
                torch.multinomial(p, num_samples=1).squeeze(-1)
                for p in probs
            ], dim=1).numpy()[0]
        state_categs = s1[:-1]
        diabetic_idx = s1[-1]
        env.state = State(
            state_categs=state_categs,
            diabetic_idx=diabetic_idx,
        )
        s = env.state
    _trajectories.append(trajectory)
    _actions_vec.append(action_vec)
    _actions_idx.append(action_idx)
    _rewards_trajectories.append(reward_traj)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as first_term_DRPPI.pkl
with open('data/first_term_DRPPI.pkl', 'wb') as f:
    pickle.dump(data, f)

Epoch [0/100], Loss T: 6.2023
Epoch [10/100], Loss T: 2.4149
Epoch [20/100], Loss T: 0.8150
Epoch [30/100], Loss T: 0.6319
Epoch [40/100], Loss T: 0.3777
Epoch [50/100], Loss T: 0.1421
Epoch [60/100], Loss T: 0.2227
Epoch [70/100], Loss T: 0.1374
Epoch [80/100], Loss T: 0.0630
Epoch [90/100], Loss T: 0.0647


In [807]:
# for each traj in cal_0_o_trajs, generate 100 trajectories M=100
M = 100
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(len(cal_0_o_trajs)):
    for _ in range(M):
        reward = 0
        trajectory = []
        action_vec = []
        action_idx = []
        reward_traj = []
        env.state = State(
            state_categs=cal_0_o_trajs[i][0][:-1],
            diabetic_idx=cal_0_o_trajs[i][0][-1],
        )
        s = env.state
        for h in range(H):
            s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
            assert h > 0 or np.array_equal(s_vec, cal_0_o_trajs[i][0])
            trajectory.append(s_vec)
            a_idx = np.random.choice(n_actions, p=pi_e)
            a = Action(action_idx=a_idx)
            record_a = a.get_action_vec().reshape(-1)
            action_vec.append(record_a)
            action_idx.append(a_idx)
            r = env.transition(a)
            reward_traj.append(r)
            reward += r * gamma**h
            if r != 0:
                break
            tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
            tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
            with torch.no_grad():
                logits = T(tensor_s, tensor_a)
                probs = [F.softmax(log, dim=-1) for log in logits]
                # sample each dimension independently
                s1 = torch.stack([
                    torch.multinomial(p, num_samples=1).squeeze(-1)
                    for p in probs
                ], dim=1).numpy()[0]
            state_categs = s1[:-1]
            diabetic_idx = s1[-1]
            env.state = State(
                state_categs=state_categs,
                diabetic_idx=diabetic_idx,
            )
            s = env.state
        _trajectories.append(trajectory)
        _actions_vec.append(action_vec)
        _actions_idx.append(action_idx)
        _rewards_trajectories.append(reward_traj)
        _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('data/second_term_matching_DRPPI.pkl', 'wb') as f:
    pickle.dump(data, f)



In [808]:
# second term for two PPI methods
second_term_matching_DRPPI = pickle.load(open("data/second_term_matching_DRPPI.pkl", "rb"))
second_term_matching_DRPPI_trajs = second_term_matching_DRPPI['trajectories']
second_term_matching_DRPPI_actions_vec = second_term_matching_DRPPI['actions_vec']
second_term_matching_DRPPI_actions_idx = second_term_matching_DRPPI['actions_idx']
second_term_matching_DRPPI_rewards_trajs = second_term_matching_DRPPI['rewards_trajectories']
second_term_matching_DRPPI_rewards = second_term_matching_DRPPI['rewards']

cal_s0_pairs = []
for i in range(len(cal_0_o_trajs)):
    for j in range(len(second_term_matching_DRPPI_trajs)):
        # check if the first state of cal_0_o_trajs[i] is the same as the first state of second_term_matching_DRPPI_trajs[j]
        # if they are the same, add the pair (i, j) to cal_s0_pairs
        if np.array_equal(cal_0_o_trajs[i][0], second_term_matching_DRPPI_trajs[j][0]):
            cal_s0_pairs.append((i, j))

print(len(cal_s0_pairs))

1400


In [826]:
# first term for DR-PPI methods
first_term_DRPPI = pickle.load(open("data/first_term_DRPPI.pkl", "rb"))
first_term_DRPPI_trajs = first_term_DRPPI['trajectories']
first_term_DRPPI_actions_vec = first_term_DRPPI['actions_vec']
first_term_DRPPI_actions_idx = first_term_DRPPI['actions_idx']
first_term_DRPPI_rewards_trajs = first_term_DRPPI['rewards_trajectories']
first_term_DRPPI_rewards = first_term_DRPPI['rewards']
first_term_DRPPI_rewards_mean = np.mean(first_term_DRPPI_rewards)
sigma_f2 = np.var(first_term_DRPPI_rewards)

#IS weights
weights = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    w = 1
    for t in range(len(cal_0_o_trajs[i])):
        w *= pi_e[int(cal_0_o_actions_idx[i][t])] / pi_b[int(cal_0_o_actions_idx[i][t])]
    weights[i] = w

E_diff_rewards_IS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = weights[i]*cal_0_o_rewards[i] - second_term_matching_DRPPI_rewards[j]
    E_diff_rewards_IS[i] += diff
E_diff_rewards_IS = E_diff_rewards_IS / M

E_diff_rewards_IS_mean = np.mean(E_diff_rewards_IS)
sigma_b2_IS = np.var(E_diff_rewards_IS)

# WIS
normalized_weights = np.zeros(len(cal_0_o_trajs))
normalized_weights = weights / np.sum(weights)
E_diff_rewards_WIS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = normalized_weights[i]*cal_0_o_rewards[i] - second_term_matching_DRPPI_rewards[j]/len(cal_0_o_trajs)
    E_diff_rewards_WIS[i] += diff
E_diff_rewards_WIS = E_diff_rewards_WIS / M
E_diff_rewards_WIS_mean = np.sum(E_diff_rewards_WIS)
sigma_b2_WIS = np.var(E_diff_rewards_WIS)

# PDIS
reweighted_values = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    w = 1
    v = 0
    for t in range(len(cal_0_o_trajs[i])):
        w *= pi_e[int(cal_0_o_actions_idx[i][t])] / pi_b[int(cal_0_o_actions_idx[i][t])]
        v += cal_0_o_rewards_trajs[i][t] * (gamma ** t) * w
    reweighted_values[i] = v

E_diff_rewards_PDIS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = reweighted_values[i] - second_term_matching_DRPPI_rewards[j]
    E_diff_rewards_PDIS[i] += diff
E_diff_rewards_PDIS = E_diff_rewards_PDIS / M
E_diff_rewards_PDIS_mean = np.mean(E_diff_rewards_PDIS)
sigma_b2_PDIS = np.var(E_diff_rewards_PDIS)

# V_DRPPI_IS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_IS_mean
# V_DRPPI_IS_var = sigma_f2/N_f + sigma_b2_IS/len(cal_0_o_trajs)
# V_DRPPI_IS_lb = V_DRPPI_IS_mean - z * np.sqrt(V_DRPPI_IS_var)
# V_DRPPI_IS_ub = V_DRPPI_IS_mean + z * np.sqrt(V_DRPPI_IS_var)
# print('V_DRPPI_IS:', V_DRPPI_IS_mean)
# print('V_DRPPI_IS_LB:', V_DRPPI_IS_lb)
# print('V_DRPPI_IS_UB:', V_DRPPI_IS_ub)

# V_DRPPI_WIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_WIS_mean
# V_DRPPI_WIS_var = sigma_f2/N_f + sigma_b2_WIS/len(cal_0_o_trajs)
# V_DRPPI_WIS_lb = V_DRPPI_WIS_mean - z * np.sqrt(V_DRPPI_WIS_var)
# V_DRPPI_WIS_ub = V_DRPPI_WIS_mean + z * np.sqrt(V_DRPPI_WIS_var)
# print('V_DRPPI_WIS:', V_DRPPI_WIS_mean)
# print('V_DRPPI_WIS_LB:', V_DRPPI_WIS_lb)
# print('V_DRPPI_WIS_UB:', V_DRPPI_WIS_ub)

# V_DRPPI_PDIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_PDIS_mean
# V_DRPPI_PDIS_var = sigma_f2/N_f + sigma_b2_PDIS/len(cal_0_o_trajs)
# V_DRPPI_PDIS_lb = V_DRPPI_PDIS_mean - z * np.sqrt(V_DRPPI_PDIS_var)
# V_DRPPI_PDIS_ub = V_DRPPI_PDIS_mean + z * np.sqrt(V_DRPPI_PDIS_var)
# print('V_DRPPI_PDIS:', V_DRPPI_PDIS_mean)
# print('V_DRPPI_PDIS_LB:', V_DRPPI_PDIS_lb)
# print('V_DRPPI_PDIS_UB:', V_DRPPI_PDIS_ub)

In [827]:
# try a cross-fit?
# train a transition model using cal_0_o
T_cal = train_T(cal_0_o, epochs=100, batch_size=16)
# generate 5000 trajectories, serve as the first term in DR-PPI N_f=5000
N_f = 1000
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(N_f):
    reward = 0
    trajectory = []
    action_vec = []
    action_idx = []
    reward_traj = []
    s = env.get_new_state(idx_type="full")
    env.state = s
    for h in range(H):
        s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        trajectory.append(s_vec)
        a_idx = np.random.choice(n_actions, p=pi_e)
        a = Action(action_idx=a_idx)
        record_a = a.get_action_vec().reshape(-1)
        action_vec.append(record_a)
        action_idx.append(a_idx)
        r = env.transition(a)
        reward_traj.append(r)
        reward += r * gamma**h
        if r != 0:
            break
        tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
        tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
        with torch.no_grad():
            logits = T_cal(tensor_s, tensor_a)
            probs = [F.softmax(log, dim=-1) for log in logits]
            # sample each dimension independently
            s1 = torch.stack([
                torch.multinomial(p, num_samples=1).squeeze(-1)
                for p in probs
            ], dim=1).numpy()[0]
        state_categs = s1[:-1]
        diabetic_idx = s1[-1]
        env.state = State(
            state_categs=state_categs,
            diabetic_idx=diabetic_idx,
        )
        s = env.state
    _trajectories.append(trajectory)
    _actions_vec.append(action_vec)
    _actions_idx.append(action_idx)
    _rewards_trajectories.append(reward_traj)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as first_term_DRPPI.pkl
with open('data/first_term_DRPPI_2.pkl', 'wb') as f:
    pickle.dump(data, f)

Epoch [0/100], Loss T: 5.9112
Epoch [10/100], Loss T: 2.1223
Epoch [20/100], Loss T: 0.7648
Epoch [30/100], Loss T: 0.5920
Epoch [40/100], Loss T: 0.5195
Epoch [50/100], Loss T: 0.4548
Epoch [60/100], Loss T: 0.5888
Epoch [70/100], Loss T: 0.4933
Epoch [80/100], Loss T: 0.1266
Epoch [90/100], Loss T: 0.2557


In [811]:
# for each traj in train_0_o_trajs, generate 100 trajectories M=100
M = 100
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(len(train_0_o_trajs)):
    for _ in range(M):
        reward = 0
        trajectory = []
        action_vec = []
        action_idx = []
        reward_traj = []
        env.state = State(
            state_categs=train_0_o_trajs[i][0][:-1],
            diabetic_idx=train_0_o_trajs[i][0][-1],
        )
        s = env.state
        for h in range(H):
            s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
            assert h > 0 or np.array_equal(s_vec, train_0_o_trajs[i][0])
            trajectory.append(s_vec)
            a_idx = np.random.choice(n_actions, p=pi_e)
            a = Action(action_idx=a_idx)
            record_a = a.get_action_vec().reshape(-1)
            action_vec.append(record_a)
            action_idx.append(a_idx)
            r = env.transition(a)
            reward_traj.append(r)
            reward += r * gamma**h
            if r != 0:
                break
            tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
            tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
            with torch.no_grad():
                logits = T_cal(tensor_s, tensor_a)
                probs = [F.softmax(log, dim=-1) for log in logits]
                # sample each dimension independently
                s1 = torch.stack([
                    torch.multinomial(p, num_samples=1).squeeze(-1)
                    for p in probs
                ], dim=1).numpy()[0]
            state_categs = s1[:-1]
            diabetic_idx = s1[-1]
            env.state = State(
                state_categs=state_categs,
                diabetic_idx=diabetic_idx,
            )
            s = env.state
        _trajectories.append(trajectory)
        _actions_vec.append(action_vec)
        _actions_idx.append(action_idx)
        _rewards_trajectories.append(reward_traj)
        _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('data/second_term_matching_DRPPI_2.pkl', 'wb') as f:
    pickle.dump(data, f)



In [812]:
# second term for two PPI methods
second_term_matching_DRPPI_2 = pickle.load(open("data/second_term_matching_DRPPI_2.pkl", "rb"))
second_term_matching_DRPPI_trajs_2 = second_term_matching_DRPPI_2['trajectories']
second_term_matching_DRPPI_actions_vec_2 = second_term_matching_DRPPI_2['actions_vec']
second_term_matching_DRPPI_actions_idx_2 = second_term_matching_DRPPI_2['actions_idx']
second_term_matching_DRPPI_rewards_trajs_2 = second_term_matching_DRPPI_2['rewards_trajectories']
second_term_matching_DRPPI_rewards_2 = second_term_matching_DRPPI_2['rewards']

train_s0_pairs = []
for i in range(len(train_0_o_trajs)):
    for j in range(len(second_term_matching_DRPPI_trajs_2)):
        # check if the first state of cal_0_o_trajs[i] is the same as the first state of second_term_matching_DRPPI_trajs[j]
        # if they are the same, add the pair (i, j) to cal_s0_pairs
        if np.array_equal(train_0_o_trajs[i][0], second_term_matching_DRPPI_trajs_2[j][0]):
            train_s0_pairs.append((i, j))

print(len(train_s0_pairs))

1200


In [832]:
# first term for DR-PPI methods
first_term_DRPPI_2 = pickle.load(open("data/first_term_DRPPI_2.pkl", "rb"))
first_term_DRPPI_trajs_2 = first_term_DRPPI_2['trajectories']
first_term_DRPPI_actions_vec_2 = first_term_DRPPI_2['actions_vec']
first_term_DRPPI_actions_idx_2 = first_term_DRPPI_2['actions_idx']
first_term_DRPPI_rewards_trajs_2 = first_term_DRPPI_2['rewards_trajectories']
first_term_DRPPI_rewards_2 = first_term_DRPPI_2['rewards']
first_term_DRPPI_rewards_mean_2 = np.mean(first_term_DRPPI_rewards_2)
sigma_f2_2 = np.var(first_term_DRPPI_rewards_2)

# IS
weights_2 = np.zeros(len(train_0_o_trajs))
for i in range(len(train_0_o_trajs)):
    w = 1
    for t in range(len(train_0_o_trajs[i])):
        w *= pi_e[int(train_0_o_actions_idx[i][t])] / pi_b[int(train_0_o_actions_idx[i][t])]
    weights_2[i] = w
E_diff_rewards_IS_2 = np.zeros(len(train_0_o_trajs))
for i, j in train_s0_pairs:  
    diff = weights_2[i]*train_0_o_rewards[i] - second_term_matching_DRPPI_rewards_2[j]
    E_diff_rewards_IS_2[i] += diff
E_diff_rewards_IS_2 = E_diff_rewards_IS_2 / M
E_diff_rewards_IS_mean_2 = np.mean(E_diff_rewards_IS_2)
sigma_b2_IS_2 = np.var(E_diff_rewards_IS_2)

V_DRPPI_IS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_IS_mean
V_DRPPI_IS_var = sigma_f2/N_f + sigma_b2_IS/len(cal_0_o_trajs)
V_DRPPI_IS_lb = V_DRPPI_IS_mean - z * np.sqrt(V_DRPPI_IS_var)
V_DRPPI_IS_ub = V_DRPPI_IS_mean + z * np.sqrt(V_DRPPI_IS_var)
print('V_DRPPI_IS:', V_DRPPI_IS_mean)
print('V_DRPPI_IS_LB:', V_DRPPI_IS_lb)
print('V_DRPPI_IS_UB:', V_DRPPI_IS_ub)

V_DRPPI_IS_mean_2 = first_term_DRPPI_rewards_mean_2 + E_diff_rewards_IS_mean_2
V_DRPPI_IS_var_2 = sigma_f2_2/N_f + sigma_b2_IS_2/len(train_0_o_trajs)
V_DRPPI_IS_lb_2 = V_DRPPI_IS_mean_2 - z * np.sqrt(V_DRPPI_IS_var_2)
V_DRPPI_IS_ub_2 = V_DRPPI_IS_mean_2 + z * np.sqrt(V_DRPPI_IS_var_2)
print('V_DRPPI_IS_2:', V_DRPPI_IS_mean_2)
print('V_DRPPI_IS_LB_2:', V_DRPPI_IS_lb_2)
print('V_DRPPI_IS_UB_2:', V_DRPPI_IS_ub_2)

V_DRPPI_IS_mean_cf = 1/2 * (first_term_DRPPI_rewards_mean + E_diff_rewards_IS_mean + first_term_DRPPI_rewards_mean_2 + E_diff_rewards_IS_mean_2)
V_DRPPI_IS_var_cf = 1/4 * (sigma_f2/N_f + sigma_b2_IS/len(cal_0_o_trajs) + sigma_f2_2/N_f + sigma_b2_IS_2/len(train_0_o_trajs))
V_DRPPI_IS_lb_cf = V_DRPPI_IS_mean_cf - z * np.sqrt(V_DRPPI_IS_var_cf)
V_DRPPI_IS_ub_cf = V_DRPPI_IS_mean_cf + z * np.sqrt(V_DRPPI_IS_var_cf)
print('V_DRPPI_IS_cf:', V_DRPPI_IS_mean_cf)
print('V_DRPPI_IS_LB_cf:', V_DRPPI_IS_lb_cf)
print('V_DRPPI_IS_UB_cf:', V_DRPPI_IS_ub_cf)

# WIS
normalized_weights_2 = np.zeros(len(train_0_o_trajs))
normalized_weights_2 = weights_2 / np.sum(weights_2)
E_diff_rewards_WIS_2 = np.zeros(len(train_0_o_trajs))
for i, j in train_s0_pairs:
    diff = normalized_weights_2[i]*train_0_o_rewards[i] - second_term_matching_DRPPI_rewards_2[j]/len(train_0_o_trajs)
    E_diff_rewards_WIS_2[i] += diff
E_diff_rewards_WIS_2 = E_diff_rewards_WIS_2 / M
E_diff_rewards_WIS_mean_2 = np.sum(E_diff_rewards_WIS_2)
sigma_b2_WIS_2 = np.var(E_diff_rewards_WIS_2)

V_DRPPI_WIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_WIS_mean
V_DRPPI_WIS_var = sigma_f2/N_f + sigma_b2_WIS*len(cal_0_o_trajs)
V_DRPPI_WIS_lb = V_DRPPI_WIS_mean - z * np.sqrt(V_DRPPI_WIS_var)
V_DRPPI_WIS_ub = V_DRPPI_WIS_mean + z * np.sqrt(V_DRPPI_WIS_var)
print('V_DRPPI_WIS:', V_DRPPI_WIS_mean)
print('V_DRPPI_WIS_LB:', V_DRPPI_WIS_lb)
print('V_DRPPI_WIS_UB:', V_DRPPI_WIS_ub)

V_DRPPI_WIS_mean_2 = first_term_DRPPI_rewards_mean_2 + E_diff_rewards_WIS_mean_2
V_DRPPI_WIS_var_2 = sigma_f2_2/N_f + sigma_b2_WIS_2*len(train_0_o_trajs)
V_DRPPI_WIS_lb_2 = V_DRPPI_WIS_mean_2 - z * np.sqrt(V_DRPPI_WIS_var_2)
V_DRPPI_WIS_ub_2 = V_DRPPI_WIS_mean_2 + z * np.sqrt(V_DRPPI_WIS_var_2)
print('V_DRPPI_WIS_2:', V_DRPPI_WIS_mean_2)
print('V_DRPPI_WIS_LB_2:', V_DRPPI_WIS_lb_2)
print('V_DRPPI_WIS_UB_2:', V_DRPPI_WIS_ub_2)

V_DRPPI_WIS_mean_cf = 1/2 * (first_term_DRPPI_rewards_mean + E_diff_rewards_WIS_mean + first_term_DRPPI_rewards_mean_2 + E_diff_rewards_WIS_mean_2)
V_DRPPI_WIS_var_cf = 1/4 * (sigma_f2/N_f + sigma_b2_WIS*len(cal_0_o_trajs) + sigma_f2_2/N_f + sigma_b2_WIS_2*len(train_0_o_trajs))
V_DRPPI_WIS_lb_cf = V_DRPPI_WIS_mean_cf - z * np.sqrt(V_DRPPI_WIS_var_cf)
V_DRPPI_WIS_ub_cf = V_DRPPI_WIS_mean_cf + z * np.sqrt(V_DRPPI_WIS_var_cf)
print('V_DRPPI_WIS_cf:', V_DRPPI_WIS_mean_cf)
print('V_DRPPI_WIS_LB_cf:', V_DRPPI_WIS_lb_cf)
print('V_DRPPI_WIS_UB_cf:', V_DRPPI_WIS_ub_cf)

# PDIS
reweighted_values = np.zeros(len(train_0_o_trajs))
for i in range(len(train_0_o_trajs)):
    w = 1
    v = 0
    for t in range(len(train_0_o_trajs[i])):
        w *= pi_e[int(train_0_o_actions_idx[i][t])] / pi_b[int(train_0_o_actions_idx[i][t])]
        v += train_0_o_rewards_trajs[i][t] * (gamma ** t) * w
    reweighted_values[i] = v

E_diff_rewards_PDIS_2 = np.zeros(len(train_0_o_trajs))
for i, j in train_s0_pairs:  
    diff = reweighted_values[i] - second_term_matching_DRPPI_rewards_2[j]
    E_diff_rewards_PDIS_2[i] += diff
E_diff_rewards_PDIS_2 = E_diff_rewards_PDIS_2 / M
E_diff_rewards_PDIS_mean_2 = np.mean(E_diff_rewards_PDIS_2)
sigma_b2_PDIS_2 = np.var(E_diff_rewards_PDIS_2)

V_DRPPI_PDIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_PDIS_mean
V_DRPPI_PDIS_var = sigma_f2/N_f + sigma_b2_PDIS/len(cal_0_o_trajs)
V_DRPPI_PDIS_lb = V_DRPPI_PDIS_mean - z * np.sqrt(V_DRPPI_PDIS_var)
V_DRPPI_PDIS_ub = V_DRPPI_PDIS_mean + z * np.sqrt(V_DRPPI_PDIS_var)
print('V_DRPPI_PDIS:', V_DRPPI_PDIS_mean)
print('V_DRPPI_PDIS_LB:', V_DRPPI_PDIS_lb)
print('V_DRPPI_PDIS_UB:', V_DRPPI_PDIS_ub)

V_DRPPI_PDIS_mean_2 = first_term_DRPPI_rewards_mean_2 + E_diff_rewards_PDIS_mean_2
V_DRPPI_PDIS_var_2 = sigma_f2_2/N_f + sigma_b2_PDIS_2/len(train_0_o_trajs)
V_DRPPI_PDIS_lb_2 = V_DRPPI_PDIS_mean_2 - z * np.sqrt(V_DRPPI_PDIS_var_2)
V_DRPPI_PDIS_ub_2 = V_DRPPI_PDIS_mean_2 + z * np.sqrt(V_DRPPI_PDIS_var_2)
print('V_DRPPI_PDIS_2:', V_DRPPI_PDIS_mean_2)
print('V_DRPPI_PDIS_LB_2:', V_DRPPI_PDIS_lb_2)
print('V_DRPPI_PDIS_UB_2:', V_DRPPI_PDIS_ub_2)

V_DRPPI_PDIS_mean_cf = 1/2 * (first_term_DRPPI_rewards_mean + E_diff_rewards_PDIS_mean + first_term_DRPPI_rewards_mean_2 + E_diff_rewards_PDIS_mean_2)
V_DRPPI_PDIS_var_cf = 1/4 * (sigma_f2/N_f + sigma_b2_PDIS/len(cal_0_o_trajs) + sigma_f2_2/N_f + sigma_b2_PDIS_2/len(train_0_o_trajs))
V_DRPPI_PDIS_lb_cf = V_DRPPI_PDIS_mean_cf - z * np.sqrt(V_DRPPI_PDIS_var_cf)
V_DRPPI_PDIS_ub_cf = V_DRPPI_PDIS_mean_cf + z * np.sqrt(V_DRPPI_PDIS_var_cf)
print('V_DRPPI_PDIS_cf:', V_DRPPI_PDIS_mean_cf)
print('V_DRPPI_PDIS_LB_cf:', V_DRPPI_PDIS_lb_cf)
print('V_DRPPI_PDIS_UB_cf:', V_DRPPI_PDIS_ub_cf)

V_DRPPI_IS: 1.3737179524038188
V_DRPPI_IS_LB: -1.4541377983021657
V_DRPPI_IS_UB: 4.201573703109803
V_DRPPI_IS_2: 0.023946216682520005
V_DRPPI_IS_LB_2: -0.13744799670833102
V_DRPPI_IS_UB_2: 0.18534043007337103
V_DRPPI_IS_cf: 0.6988320845431694
V_DRPPI_IS_LB_cf: -0.7173967316194022
V_DRPPI_IS_UB_cf: 2.115060900705741
V_DRPPI_WIS: 1.5696538654650043
V_DRPPI_WIS_LB: -1.64279687021316
V_DRPPI_WIS_UB: 4.7821046011431685
V_DRPPI_WIS_2: -0.827799027969754
V_DRPPI_WIS_LB_2: -2.0229387586670997
V_DRPPI_WIS_UB_2: 0.3673407027275919
V_DRPPI_WIS_cf: 0.3709274187476251
V_DRPPI_WIS_LB_cf: -1.3428548533810798
V_DRPPI_WIS_UB_cf: 2.08470969087633
V_DRPPI_PDIS: 1.3737179524038188
V_DRPPI_PDIS_LB: -1.4541377983021657
V_DRPPI_PDIS_UB: 4.201573703109803
V_DRPPI_PDIS_2: 0.023946216682520005
V_DRPPI_PDIS_LB_2: -0.13744799670833102
V_DRPPI_PDIS_UB_2: 0.18534043007337103
V_DRPPI_PDIS_cf: 0.6988320845431694
V_DRPPI_PDIS_LB_cf: -0.7173967316194022
V_DRPPI_PDIS_UB_cf: 2.115060900705741


In [814]:
# CP-PPI
# for each traj in train_0_o_trajs, generate M trajectories
M = 100
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(len(train_0_o_trajs)):
    for _ in range(M):
        reward = 0
        trajectory = []
        action_vec = []
        action_idx = []
        reward_traj = []
        env.state = State(
            state_categs=train_0_o_trajs[i][0][:-1],
            diabetic_idx=train_0_o_trajs[i][0][-1],
        )
        s = env.state
        for h in range(H):
            s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
            assert h > 0 or np.array_equal(s_vec, train_0_o_trajs[i][0])
            trajectory.append(s_vec)
            a_idx = np.random.choice(n_actions, p=pi_b)
            a = Action(action_idx=a_idx)
            record_a = a.get_action_vec().reshape(-1)
            action_vec.append(record_a)
            action_idx.append(a_idx)
            r = env.transition(a)
            reward_traj.append(r)
            reward += r * gamma**h
            if r != 0:
                break
            tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
            tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
            with torch.no_grad():
                logits = T(tensor_s, tensor_a)
                probs = [F.softmax(log, dim=-1) for log in logits]
                # sample each dimension independently
                s1 = torch.stack([
                    torch.multinomial(p, num_samples=1).squeeze(-1)
                    for p in probs
                ], dim=1).numpy()[0]
            state_categs = [ s1[i] for i in range(len(state_dims)-1) ]
            diabetic_idx = s1[-1]
            env.state = State(
                state_categs=state_categs,
                diabetic_idx=diabetic_idx,
            )
            s = env.state
        _trajectories.append(trajectory)
        _actions_vec.append(action_vec)
        _actions_idx.append(action_idx)
        _rewards_trajectories.append(reward_traj)
        _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('data/train_matching_CPPPI.pkl', 'wb') as f:
    pickle.dump(data, f)



In [815]:
# for each traj in cal_0_o_trajs, generate M trajectories
M = 100
_trajectories = []
_actions_vec = []
_actions_idx = []
_rewards_trajectories = []
_rewards = []
for i in range(len(cal_0_o_trajs)):
    for _ in range(M):
        reward = 0
        trajectory = []
        action_vec = []
        action_idx = []
        reward_traj = []
        env.state = State(
            state_categs=cal_0_o_trajs[i][0][:-1],
            diabetic_idx=cal_0_o_trajs[i][0][-1],
        )
        s = env.state
        for h in range(H):
            s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
            assert h > 0 or np.array_equal(s_vec, cal_0_o_trajs[i][0])
            trajectory.append(s_vec)
            a_idx = np.random.choice(n_actions, p=pi_b)
            a = Action(action_idx=a_idx)
            record_a = a.get_action_vec().reshape(-1)
            action_vec.append(record_a)
            action_idx.append(a_idx)
            r = env.transition(a)
            reward_traj.append(r)
            reward += r * gamma**h
            if r != 0:
                break
            tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
            tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
            with torch.no_grad():
                logits = T(tensor_s, tensor_a)
                probs = [F.softmax(log, dim=-1) for log in logits]
                # sample each dimension independently
                s1 = torch.stack([
                    torch.multinomial(p, num_samples=1).squeeze(-1)
                    for p in probs
                ], dim=1).numpy()[0]
            state_categs = [ s1[i] for i in range(len(state_dims)-1) ]
            diabetic_idx = s1[-1]
            env.state = State(
                state_categs=state_categs,
                diabetic_idx=diabetic_idx,
            )
            s = env.state
        _trajectories.append(trajectory)
        _actions_vec.append(action_vec)
        _actions_idx.append(action_idx)
        _rewards_trajectories.append(reward_traj)
        _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.object_(_trajectories),
    'actions_vec': np.object_(_actions_vec),
    'actions_idx': np.object_(_actions_idx),
    'rewards_trajectories': np.object_(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('data/cal_matching_CPPPI.pkl', 'wb') as f:
    pickle.dump(data, f)

In [816]:
# CP-PPI
train_matching_PPI = pickle.load(open("data/train_matching_CPPPI.pkl", "rb"))
train_matching_PPI_trajs = train_matching_PPI['trajectories']
train_matching_PPI_actions_vec = train_matching_PPI['actions_vec']
train_matching_PPI_actions_idx = train_matching_PPI['actions_idx']
train_matching_PPI_rewards_trajs = train_matching_PPI['rewards_trajectories']
train_matching_PPI_rewards = train_matching_PPI['rewards']

cal_matching_PPI = pickle.load(open("data/cal_matching_CPPPI.pkl", "rb"))
cal_matching_PPI_trajs = cal_matching_PPI['trajectories']
cal_matching_PPI_actions_vec = cal_matching_PPI['actions_vec']
cal_matching_PPI_actions_idx = cal_matching_PPI['actions_idx']
cal_matching_PPI_rewards_trajs = cal_matching_PPI['rewards_trajectories']
cal_matching_PPI_rewards = cal_matching_PPI['rewards']

train_s0_pairs = []
for i in range(len(train_0_o_trajs)):
    for j in range(len(train_matching_PPI_trajs)):
        if np.array_equal(train_0_o_trajs[i][0], train_matching_PPI_trajs[j][0]):
            train_s0_pairs.append((i, j))

cal_s0_pairs = []
for i in range(len(cal_0_o_trajs)):
    for j in range(len(cal_matching_PPI_trajs)):
        if np.array_equal(cal_0_o_trajs[i][0], cal_matching_PPI_trajs[j][0]):
            cal_s0_pairs.append((i, j))

train_diff_rewards = []
for i, j in train_s0_pairs:
    diff = train_0_o_rewards[i] - train_matching_PPI_rewards[j]
    train_diff_rewards.append((i, j, diff)) # state = train_0_o_trajs[i]

cal_diff_rewards = []
for i, j in cal_s0_pairs:
    diff = cal_0_o_rewards[i] - cal_matching_PPI_rewards[j]
    cal_diff_rewards.append((i, j, diff)) # state = cal_0_o_trajs[i]

In [817]:
len(train_s0_pairs), len(cal_s0_pairs)

(1200, 1400)

In [818]:
eps_s = 2
eps_r = 30

matching_pairs = []
for m in range(len(cal_diff_rewards)):
    i, j, diff = cal_diff_rewards[m]
    for n in range(len(train_diff_rewards)):
        i2, j2, diff2 = train_diff_rewards[n]
        if np.linalg.norm(cal_0_o_trajs[i][0] - train_0_o_trajs[i2][0]) < eps_s and \
            abs(diff - diff2) < eps_r:
            matching_pairs.append((m, n)) # train_n can be used to calculate the weights for cal_m
print(len(matching_pairs))

1110000


In [819]:
weights = []
for m in range(len(cal_diff_rewards)):
    w = 0
    n = 0
    for m1, n1 in matching_pairs:
        w1 = 1
        if m == m1:
            i2, j2, diff2 = train_diff_rewards[n1]
            for t in range(len(train_0_o_trajs[i2])):
                w1 *= pi_e[int(train_0_o_actions_idx[i2][t])] / pi_b[int(train_0_o_actions_idx[i2][t])]
            for t in range(len(train_matching_PPI_trajs[j2])):
                w1 *= pi_e[int(train_matching_PPI_actions_idx[j2][t])] / pi_b[int(train_matching_PPI_actions_idx[j2][t])]
            w += w1
            n += 1
    if n==0:
        print('no matching pairs')
        i, j, diff = cal_diff_rewards[m]
        print(cal_0_o_trajs[i][0], cal_matching_PPI_trajs[j][0], diff)
        break
    weights.append(w/n)

# sort the weights and delta_reward
weights = np.array(weights, dtype=np.float32)
delta_reward = np.array([diff for _, _, diff in cal_diff_rewards], dtype=np.float32)
delta_reward_sorted = np.sort(delta_reward)
weights_sorted = weights[np.argsort(delta_reward)]

In [820]:
def w(x, y, eps_s, eps_r):
    matching_pairs = []
    for i1, j1 in train_s0_pairs:
        if np.linalg.norm(train_0_o_trajs[i1][0] - x) < eps_s and \
            abs((train_0_o_rewards[i1] - train_matching_PPI_rewards[j1]) - y) < eps_r:
            matching_pairs.append((i1, j1))
    w = 0
    n = len(matching_pairs)
    for i, j in matching_pairs:
        w1 = 1
        for t in range(len(train_0_o_trajs[i])):
            w1 *= pi_e[int(train_0_o_actions_idx[i][t])] / pi_b[int(train_0_o_actions_idx[i][t])]
        for t in range(len(train_matching_PPI_trajs[j])):
            w1 *= pi_e[int(train_matching_PPI_actions_idx[j][t])] / pi_b[int(train_matching_PPI_actions_idx[j][t])]
        w += w1
    if n > 0:
        return w/n
    else:
        return -1
    
start_s = cal_0_o_trajs[0][0]
y_list = np.linspace(-100, 100, 100)
CP_s_y = []
for y in y_list:
    w_ = w(start_s, y, eps_s, eps_r)
    if w_ != -1:
        new_weights = np.append(weights_sorted, w_)
        normalized_weights = new_weights / np.sum(new_weights)
        
        low_level = 0
        high_level = 0
        low_q = 0
        high_q = 0
        for i in range(len(delta_reward_sorted)):
            low_level += normalized_weights[i]
            if low_level >= alpha/2:
                low_q = delta_reward_sorted[i-1]
                break
        for i in range(len(delta_reward_sorted)):
            high_level += normalized_weights[i]
            if high_level >= 1 - alpha/2:
                high_q = delta_reward_sorted[i-1]
                break
        print("y: ", y, "low_q: ", low_q, "high_q: ", high_q)
        break
        # if low_q <= y <= high_q:
        #     CP_1_y.append(y)

y:  -31.313131313131308 low_q:  -1.0 high_q:  0.71469796


In [821]:
_rewards = []
for _ in range(500):
    s = State(
        state_categs=start_s[:-1],
        diabetic_idx=start_s[-1],
    )
    env.state = s
    reward = 0
    for h in range(H):
        s_vec = np.concatenate((s.get_state_vector(), [s.diabetic_idx]))
        a_idx = np.random.choice(n_actions, p=pi_e)
        a = Action(action_idx=a_idx)
        r = env.transition(a)
        reward += r * gamma**h
        if r != 0:
            break
        tensor_s = torch.LongTensor(s_vec.reshape(1, -1))
        tensor_a = torch.FloatTensor(a.get_action_vec().reshape(1, -1))
        with torch.no_grad():
            logits = T(tensor_s, tensor_a)
            probs = [F.softmax(log, dim=-1) for log in logits]
            # sample each dimension independently
            s1 = torch.stack([
                torch.multinomial(p, num_samples=1).squeeze(-1)
                for p in probs
            ], dim=1).numpy()[0]
        state_categs = [ s1[i] for i in range(len(state_dims)-1) ]
        diabetic_idx = s1[-1]
        env.state = State(
            state_categs=state_categs,
            diabetic_idx=diabetic_idx,
        )
        s = env.state
    _rewards.append(reward)

hat_V_pi_e = np.mean(_rewards)
CP = (hat_V_pi_e + low_q, hat_V_pi_e + high_q)
print("hat_V_pi_e: ", hat_V_pi_e)
print("CP: ", CP)

hat_V_pi_e:  -0.37440569801704177
CP:  (np.float64(-1.3744056980170418), np.float64(0.34029225902183763))


In [822]:
cal_0_o_trajs[0][0]

array([0, 1, 1, 2, 0, 0, 0, 0])

In [825]:
# evaluate the policies by MC for 1000 episodes
def evaluate_policy(policy, gamma, num_episodes=1000):
    total_reward = 0
    for _ in range(num_episodes):
        # s = env.get_new_state(idx_type="full")
        s = State(
            state_categs=start_s[:-1],
            diabetic_idx=start_s[-1],
        )
        env.state = s
        for t in range(H):
            a = np.random.choice(n_actions, p=policy)
            a = Action(action_idx=a)
            r = env.transition(a)
            total_reward += r * gamma**t
            if r != 0:
                break
            s = env.state
    return total_reward / num_episodes


evaluate_policy(pi_e, gamma=gamma, num_episodes=50000)

-0.46029534531343336